# **06_Random_Forest.py**

Using the same train/test splits as in 05_Logistic_Regression.py, Random Forest is trained on both datasets for a similar comparison.

Inputs: elliptic_train.csv, elliptic_test.csv, and ethereum_train.csv.


*   elliptic_train.csv
*   elliptic_test.csv
*   ethereum_train.csv

Outputs: rf_elliptic.joblib, rf_ethereum.joblib, and rf_results.json.


*   rf_elliptic.joblib
*   rf_ethereum.joblib
*   rf_results.json

In [35]:
import os
import json
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, roc_auc_score, confusion_matrix)

In [36]:
BASE_PATH = ".."

DATA_PATH = os.path.join(BASE_PATH, "cleaned_data")

MODEL_PATH = os.path.join(BASE_PATH, "models")

os.makedirs(MODEL_PATH, exist_ok=True)

results = {}

In [37]:
def evaluate(model, X_test, y_test):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, prob),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }

In [38]:
RF_PARAMS = dict(n_estimators=200, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1)

In [39]:
# ELLIPTIC DATASET
ell_train = pd.read_csv("../train_test_data/elliptic_train.csv")
ell_test = pd.read_csv("../train_test_data/elliptic_test.csv")
feat_cols = [c for c in ell_train.columns if c.startswith("feat_")]

In [40]:
X_train, y_train = ell_train[feat_cols], ell_train["label"]
X_test, y_test = ell_test[feat_cols], ell_test["label"]

In [41]:
rf_ell = RandomForestClassifier(**RF_PARAMS)
rf_ell.fit(X_train, y_train)
results["elliptic_rf"] = evaluate(rf_ell, X_test, y_test)

In [42]:
# ETHEREUM DATASET
eth_train = pd.read_csv("../train_test_data/ethereum_train.csv")
eth_test = pd.read_csv("../train_test_data/ethereum_test.csv")
feat_cols_eth = [c for c in eth_train.columns if c not in ["Address", "FLAG"]]

In [43]:
X_train_e, y_train_e = eth_train[feat_cols_eth], eth_train["FLAG"]
X_test_e, y_test_e = eth_test[feat_cols_eth], eth_test["FLAG"]

In [44]:


rf_eth = RandomForestClassifier(**RF_PARAMS)
rf_eth.fit(X_train_e, y_train_e)
results["ethereum_rf"] = evaluate(rf_eth, X_test_e, y_test_e)


In [45]:
for name, r in results.items():
    print(f"\n--- {name} ---")
    for k, v in r.items():
        if k != "confusion_matrix":
            print(f"{k:>10}: {v:.4f}")
    print("confusion matrix:", r["confusion_matrix"])


--- elliptic_rf ---
  accuracy: 0.9686
 precision: 0.7718
    recall: 0.7341
        f1: 0.7525
   roc_auc: 0.9349
confusion matrix: [[15352, 235], [288, 795]]

--- ethereum_rf ---
  accuracy: 0.9791
 precision: 0.9853
    recall: 0.9197
        f1: 0.9514
   roc_auc: 0.9969
confusion matrix: [[1522, 6], [35, 401]]


In [46]:
# Save trained Random Forest models
joblib.dump(
    rf_ell,
    "../models/rf_elliptic.joblib"
)

joblib.dump(
    rf_eth,
    "../models/rf_ethereum.joblib"
)


['../models/rf_ethereum.joblib']

In [47]:
# Save evaluation results
with open(
    "../models/rf_results.json",
    "w"
) as f:
    json.dump(results, f, indent=2)

print("Random Forest files saved successfully!")

Random Forest files saved successfully!
